# Market Basket & Customer Behaviour Analysis

This notebook builds on the
[Instacart Online Grocery Shopping Dataset 2017](https://www.instacart.com/datasets/grocery-shopping-2017)
and goes one step beyond `eda.ipynb` and `analysis.ipynb`. It covers two
related but distinct analytical themes:

### Part A — Market Basket Analysis
*"Which products tend to be bought together?"* We answer this question two
ways:

1. **Co-occurrence counting** — a simple, direct count of how often each
   product (or aisle) pair appears in the same order.
2. **Association-rule mining with Apriori** — a more formal approach that
   produces interpretable *if–then* rules along with the standard support,
   confidence, and lift metrics.

### Part B — Customer Behaviour Analysis
*"What kinds of customers does Instacart have, and how do they shop
differently?"* We build a per-user profile (orders, basket size, reorder
rate, cadence), then **segment** customers into Light / Regular / Frequent /
Power groups and compare their habits.

### Notebook structure

Every analysis step follows the same pattern: a short markdown cell sets
up *what* we're about to compute and *why*, the code cell does the work,
and a follow-up markdown cell reads off the takeaway. All charts use Plotly
Express with the `"simple_white"` template.


## Set up the environment

We use **pandas/NumPy** for data wrangling, **Plotly Express** for
visualisation, and **mlxtend** (loaded later) for the Apriori algorithm.
We silence pandas' chained-assignment / `FutureWarning` noise so the
output stays readable.


In [1]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

px.defaults.template = "simple_white"

### Load the data

The dataset is split across five CSV files. We pull them straight from
the public mirror on GitHub. The orders file is shipped as a ZIP archive,
so we tell pandas to decompress it on the fly.


In [ ]:
# Load all five tables
base_url = "https://raw.githubusercontent.com/subwaymatch/instacart-dataset-2017/refs/heads/main/"

orders = pd.read_csv(base_url + "orders.csv.zip", compression="zip")
order_products = pd.read_csv(base_url + "order_products_train.csv")
products = pd.read_csv(base_url + "products.csv")
aisles = pd.read_csv(base_url + "aisles.csv")
departments = pd.read_csv(base_url + "departments.csv")

print(f"orders:          {orders.shape}")
print(f"order_products:  {order_products.shape}")
print(f"products:        {products.shape}")

### Build the enriched item-level table

`order_products` only carries numeric IDs. To analyse behaviour at the
product, aisle, or department level we join in the lookup tables. The
result, **`op`**, has *one row per item bought* with human-readable names
attached.


In [ ]:
# Enriched order-products table: one row per item with names attached
op = (
    order_products.merge(products, on="product_id")
    .merge(aisles, on="aisle_id")
    .merge(departments, on="department_id")
)
print(f"enriched op: {op.shape}")
op.head()

### Add order-level metadata for customer-behaviour analysis

For Part B we need to know **which user** placed each order and **when**
they placed it. We attach those fields from the `orders` table to produce
**`op_full`** — an item-level table with both product *and* customer/timing
context. This is the table we'll group by `user_id` to build per-customer
profiles.


In [ ]:
# Link order metadata (user, timing, sequence) onto each item row
op_full = op.merge(
    orders[
        [
            "order_id",
            "user_id",
            "order_dow",
            "order_hour_of_day",
            "order_number",
            "days_since_prior_order",
        ]
    ],
    on="order_id",
)
print(f"op_full: {op_full.shape}")

---

## Part A — Market Basket Analysis

Market basket analysis answers the question **"which products tend to
appear in the same order?"** The answer drives many downstream decisions:

- **Cross-selling** — recommend item B on the page of item A if they often
  go together.
- **Bundling and promotions** — discount A when bought with B.
- **Layout and search** — surface related items together in the app.

We tackle the question two ways:

1. **Co-occurrence counting** — simple, transparent, no parameters. We
   literally count how many times each pair of products (or aisles)
   appears in the same basket.
2. **Association rules with Apriori** — adds the formal metrics
   *support*, *confidence*, and *lift* that let us compare the *strength*
   of relationships, not just their raw frequency.


### Top 15 most frequently co-purchased product pairs

We build a **basket** for each order — the set of products it contains —
then enumerate all unordered pairs and count how often each pair occurs.

> ⚠️ With ~50,000 products, the full pair space is enormous (~1.25 *billion*
> possible pairs). To keep the loop fast on a laptop we restrict the
> analysis to the **top-100 most-ordered products**. This catches almost
> all of the strongest pairs because rare products simply can't co-occur
> often enough to make the top of any list.


In [ ]:
from itertools import combinations

# Restrict to the 100 most-ordered products to keep the pair loop tractable
top100 = op["product_name"].value_counts().head(100).index
op_top = op[op["product_name"].isin(top100)]

# Build one basket (a set of product names) per order
baskets = op_top.groupby("order_id")["product_name"].apply(set)
print(f"Baskets considered: {len(baskets):,}")

For every basket of size ≥ 2 we generate every unordered pair with
`itertools.combinations` and tally it in a dictionary keyed by the
(sorted) pair.


In [ ]:
# Count co-occurrences across all baskets
pair_counts = {}
for basket in baskets:
    if len(basket) < 2:
        continue
    for pair in combinations(sorted(basket), 2):
        pair_counts[pair] = pair_counts.get(pair, 0) + 1

print(f"Distinct pairs observed: {len(pair_counts):,}")

Convert the dictionary into a tidy DataFrame, sort by count, and keep
the top 15 for plotting.


In [ ]:
pairs_df = (
    pd.DataFrame(
        [(p[0], p[1], c) for p, c in pair_counts.items()],
        columns=["Product A", "Product B", "Co-occurrence Count"],
    )
    .sort_values("Co-occurrence Count", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
pairs_df["Pair"] = pairs_df["Product A"] + "  +  " + pairs_df["Product B"]
pairs_df

In [ ]:
fig = px.bar(
    pairs_df,
    x="Co-occurrence Count",
    y="Pair",
    orientation="h",
    title="Top 15 Most Frequently Co-Purchased Product Pairs",
    text_auto=True,
 height=600)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Takeaway:** Almost every top pair includes **bananas** — by far the
> most-ordered product in the dataset. This illustrates a classic pitfall
> of raw co-occurrence: a single very-popular item dominates the
> leaderboard simply because it appears in so many baskets, not because
> the relationship is unusually strong. The Apriori analysis below
> corrects for this by using **lift** to measure *unexpected* affinity.


### Top 15 most frequently co-purchased aisle pairs

Aggregating up from products to **aisles** has two big advantages:

1. There are only ~134 aisles, so we can use *every* order — no need to
   restrict to a top-N list.
2. Aisle-level relationships generalise better. "Fresh fruits + fresh
   vegetables" is a more actionable insight than any single product pair.

The recipe is identical to the product-pair analysis: build per-order
baskets of aisles, count pairs with `combinations`, sort, and plot.


In [ ]:
# Build aisle-level baskets (sets of aisles per order)
aisle_baskets = op.groupby("order_id")["aisle"].apply(set)

aisle_pair_counts = {}
for basket in aisle_baskets:
    if len(basket) < 2:
        continue
    for pair in combinations(sorted(basket), 2):
        aisle_pair_counts[pair] = aisle_pair_counts.get(pair, 0) + 1

aisle_pairs_df = (
    pd.DataFrame(
        [(p[0], p[1], c) for p, c in aisle_pair_counts.items()],
        columns=["Aisle A", "Aisle B", "Co-occurrence Count"],
    )
    .sort_values("Co-occurrence Count", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
aisle_pairs_df["Pair"] = aisle_pairs_df["Aisle A"] + "  +  " + aisle_pairs_df["Aisle B"]
aisle_pairs_df

In [ ]:
fig = px.bar(
    aisle_pairs_df,
    x="Co-occurrence Count",
    y="Pair",
    orientation="h",
    title="Top 15 Most Frequently Co-Purchased Aisle Pairs",
    text_auto=True,
 height=600)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Takeaway:** *Fresh fruits* and *fresh vegetables* sit at the top —
> unsurprisingly, since both are also the highest-volume aisles. *Yogurt*,
> *packaged vegetables*, *milk*, and *water/sparkling/seltzer* round out
> the regulars-and-essentials shopping list that defines a typical
> Instacart basket.


### Mining association rules with Apriori

Co-occurrence counts have one big weakness: a pair that contains a very
popular item can rank highly without there being any *real* affinity.
**Association rules** fix this by reporting three complementary metrics:

| Metric | Question it answers |
| ------ | ------------------- |
| **Support** | How often does this combination appear overall? *(`P(A ∩ B)`)* |
| **Confidence** | When *A* is in the basket, how often is *B* also there? *(`P(B \| A)`)* |
| **Lift** | How much more likely are *A* and *B* to co-occur than if they were independent? *(`confidence ÷ P(B)`)* — values > 1 indicate genuine affinity. |

The **Apriori algorithm** (via `mlxtend`) efficiently finds all itemsets
that meet a minimum support threshold, then derives rules from them.
We work at the **aisle level** — it's faster, easier to interpret, and
the resulting rules are usually more actionable than product-level rules.


**Step 1 — Encode each order as a binary "basket vector".** `mlxtend`'s
`TransactionEncoder` turns a list of baskets into a one-hot DataFrame
where each row is an order and each column is an aisle (`True` if that
aisle was in the order, `False` otherwise).


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Each order becomes a list of the aisles it contains
aisle_basket_list = op.groupby("order_id")["aisle"].apply(list).tolist()

te = TransactionEncoder()
te_array = te.fit(aisle_basket_list).transform(aisle_basket_list)
basket_df = pd.DataFrame(te_array, columns=te.columns_)
print(f"basket_df shape: {basket_df.shape}  (orders × aisles)")
basket_df.head()

**Step 2 — Find frequent aisle-sets with Apriori.** We require a minimum
support of **0.02**, i.e. an itemset must appear in at least 2 % of all
orders to be considered. This filters out noise while still surfacing
hundreds of itemsets.


In [ ]:
frequent_items = apriori(basket_df, min_support=0.02, use_colnames=True)
print(f"Frequent itemsets found: {len(frequent_items)}")
frequent_items.head()

**Step 3 — Derive association rules and rank them by lift.** A `lift`
threshold of 1.0 keeps only rules where the antecedent and consequent
are positively associated.


In [ ]:
rules = association_rules(frequent_items, metric="lift", min_threshold=1.0)
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)
print(f"Association rules found: {len(rules)}")
rules.head(10)

We format antecedents and consequents as readable `"A, B → C"` strings
and plot the top-20 rules ranked by **lift** (the strength of the
association beyond chance).


In [ ]:
top_rules = rules.head(20).copy()
top_rules["antecedents_str"] = top_rules["antecedents"].apply(
    lambda x: ", ".join(sorted(x))
)
top_rules["consequents_str"] = top_rules["consequents"].apply(
    lambda x: ", ".join(sorted(x))
)
top_rules["Rule"] = (
    top_rules["antecedents_str"] + "  →  " + top_rules["consequents_str"]
)

fig = px.bar(
    top_rules,
    x="lift",
    y="Rule",
    orientation="h",
    title="Top 20 Association Rules by Lift",
    text_auto=".2f",
    hover_data=["support", "confidence"],
 height=600)
fig.update_layout(yaxis=dict(autorange="reversed"), xaxis_title="Lift")
fig.show()

> **Takeaway:** The strongest rules tend to involve **complementary
> categories** — e.g. *baby food formula → baby accessories*, or
> *organic produce ↔ fresh herbs*. These are excellent candidates for
> bundle promotions or cross-aisle recommendations: when one shows up in
> the cart, the other is far more likely than chance to also belong there.


A single bar chart hides how the *three* metrics interact. The scatter
plot below shows every rule with **support on the x-axis**, **confidence
on the y-axis**, and **lift encoded by both bubble size and colour**.

Look for points in the **top-right that are also large/bright** — those
are rules that are simultaneously *frequent*, *reliable*, and *unexpectedly
strong*: the most actionable combination.


In [ ]:
rules_viz = rules.copy()
rules_viz["antecedents_str"] = rules_viz["antecedents"].apply(
    lambda x: ", ".join(sorted(x))
)
rules_viz["consequents_str"] = rules_viz["consequents"].apply(
    lambda x: ", ".join(sorted(x))
)
rules_viz["Rule"] = rules_viz["antecedents_str"] + " → " + rules_viz["consequents_str"]

fig = px.scatter(
    rules_viz,
    x="support",
    y="confidence",
    size="lift",
    color="lift",
    hover_name="Rule",
    title="Association Rules — Support vs Confidence (size & colour = Lift)",
    labels={"support": "Support", "confidence": "Confidence", "lift": "Lift"},
    color_continuous_scale="Viridis",
)
fig.show()

> **Takeaway:** Most rules cluster at low support and moderate confidence
> — the long tail of niche affinities. A handful of points sit higher and
> brighter, marking the rare combinations that are both common *and*
> strongly associated. These are the rules a real recommender system
> should prioritise.


---

## Part B — Customer Behaviour Analysis

In Part A we asked *what people buy together*. In Part B we ask
*who is doing the buying*. We summarise each user's history into a
small set of behavioural features, then **segment** customers by order
frequency and compare how the segments shop differently.

The features we engineer per customer are deliberately simple and
interpretable:

| Feature | Why it matters |
| ------- | -------------- |
| `total_orders` | Engagement / lifetime value proxy |
| `avg_basket_size` | Trip type (top-up vs full shop) |
| `reorder_rate` | Loyalty / habit strength |
| `avg_days_between_orders` | Cadence / retention risk |


### Build the per-customer profile table

We combine **three different aggregations** into a single `customers`
DataFrame — one row per `user_id`. Building it in three steps makes the
logic easier to follow and easier to debug.


**Step 1 — Order-level statistics.** Count each customer's orders and
average their cadence and timing directly from the `orders` table.


In [ ]:
user_orders = (
    orders.groupby("user_id")
    .agg(
        total_orders=("order_id", "nunique"),
        avg_days_between_orders=("days_since_prior_order", "mean"),
        avg_dow=("order_dow", "mean"),
        avg_hour=("order_hour_of_day", "mean"),
    )
    .reset_index()
)
user_orders.head()

**Step 2 — Average basket size.** First find the size of each *order*,
then average those order-sizes per *user*.


In [ ]:
basket_sizes = (
    op_full.groupby(["user_id", "order_id"]).size().reset_index(name="basket_size")
)
user_basket = (
    basket_sizes.groupby("user_id")["basket_size"]
    .mean()
    .reset_index(name="avg_basket_size")
)
user_basket.head()

**Step 3 — Reorder rate per customer.** A simple mean of the binary
`reordered` flag across all of a user's items: 0 means they never repeat,
1 means every item is a repeat purchase.


In [ ]:
user_reorder = (
    op_full.groupby("user_id")["reordered"].mean().reset_index(name="reorder_rate")
)
user_reorder.head()

**Step 4 — Combine.** Left-join the three tables on `user_id` so we keep
every customer even if their basket-size or reorder-rate fields are
missing (e.g. their orders weren't in the train split).


In [ ]:
customers = user_orders.merge(user_basket, on="user_id", how="left").merge(
    user_reorder, on="user_id", how="left"
)
print(f"customers table: {customers.shape}")
customers.head()

**Sanity check — distributional summary.** A quick `.describe()` confirms
the ranges look reasonable before we start segmenting.


In [ ]:
customers[["total_orders", "avg_basket_size", "reorder_rate", "avg_days_between_orders"]].describe()

### Distribution of Customer Lifetime Orders


How engaged is the average Instacart customer? Plotting the distribution
of `total_orders` reveals whether the user base is dominated by
occasional shoppers or by repeat regulars.


In [9]:
fig = px.histogram(
    customers,
    x="total_orders",
    nbins=100,
    title="Distribution of Total Orders per Customer",
    labels={"total_orders": "Total Orders", "count": "Number of Customers"},
)
fig.show()

> **Takeaway:** The distribution is heavily right-skewed: most customers
> have placed only a handful of orders, with a small but committed power-user
> tail. This shape motivates the frequency-based segmentation later in the
> notebook — averages alone would be very misleading for such a skewed
> population.


### Distribution of Average Basket Size per Customer


The *average* basket size *per customer* tells us about shopping style —
whether someone tends toward small top-ups or full weekly shops.


In [10]:
fig = px.histogram(
    customers.dropna(subset=["avg_basket_size"]),
    x="avg_basket_size",
    nbins=50,
    title="Distribution of Average Basket Size per Customer",
    labels={"avg_basket_size": "Avg Items per Order", "count": "Number of Customers"},
)
fig.show()

> **Takeaway:** Average basket size centres around **8–12 items** with a
> long right tail of large-basket shoppers. There is no obvious bimodality,
> so basket size on its own probably won't separate customer types — we'll
> need to combine it with frequency and reorder rate.


### Distribution of Customer Reorder Rates


Reorder rate per customer is the share of items they buy that they had
purchased before. Higher values indicate more habitual / loyal shoppers.


In [11]:
fig = px.histogram(
    customers.dropna(subset=["reorder_rate"]),
    x="reorder_rate",
    nbins=50,
    title="Distribution of Customer Reorder Rates",
    labels={"reorder_rate": "Reorder Rate", "count": "Number of Customers"},
)
fig.show()

> **Takeaway:** Most customers have reorder rates between **0.4 and 0.8** —
> repeat purchasing is the rule, not the exception, on Instacart. A small
> tail of low-reorder customers represents *exploratory* shoppers (or
> simply newer accounts with little history).


### Customer Segments by Order Frequency

We split customers into groups based on how many total orders they have placed.


We define four segments using simple, business-friendly thresholds on
`total_orders`. The thresholds (5 / 15 / 30) are chosen to give each
segment a meaningful share of the population while still telling a clear
behavioural story.


In [12]:
def frequency_segment(n):
    if n <= 5:
        return "Light (1-5)"
    elif n <= 15:
        return "Regular (6-15)"
    elif n <= 30:
        return "Frequent (16-30)"
    else:
        return "Power (31+)"


customers["frequency_segment"] = customers["total_orders"].apply(frequency_segment)

seg_order = ["Light (1-5)", "Regular (6-15)", "Frequent (16-30)", "Power (31+)"]
seg_counts = (
    customers["frequency_segment"].value_counts().reindex(seg_order).reset_index()
)
seg_counts.columns = ["Segment", "Number of Customers"]

fig = px.bar(
    seg_counts,
    x="Segment",
    y="Number of Customers",
    title="Customer Segments by Order Frequency",
    text_auto=True,
    color="Segment",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

> **Takeaway:** *Light* shoppers form the largest segment by count, but
> *Regular* and above will dominate the **revenue** and **engagement**
> picture because they each place many more orders. We'll see this in the
> per-segment averages on the next chart.


### Average Metrics by Customer Segment


Aggregate the four behavioural features by segment to get a one-row-per-segment
profile. This is the core "who are these segments?" summary.


In [13]:
seg_profile = (
    customers.groupby("frequency_segment")
    .agg(
        num_customers=("user_id", "count"),
        avg_orders=("total_orders", "mean"),
        avg_basket_size=("avg_basket_size", "mean"),
        avg_reorder_rate=("reorder_rate", "mean"),
        avg_days_between=("avg_days_between_orders", "mean"),
    )
    .reindex(seg_order)
    .reset_index()
)
seg_profile.columns = [
    "Segment",
    "Customers",
    "Avg Orders",
    "Avg Basket Size",
    "Avg Reorder Rate",
    "Avg Days Between Orders",
]
seg_profile

,Segment,Customers,Avg Orders,Avg Basket Size,Avg Reorder Rate,Avg Days Between Orders
0,Light (1-5),43576,4.449559,9.857241,0.445876,19.960406
1,Regular (6-15),92744,9.450466,10.462198,0.572717,17.246415
2,Frequent (16-30),40706,21.564290,10.939389,0.699687,12.428990
3,Power (31+),29183,50.471816,11.337511,0.794640,7.210621


> **Reading the table:** moving from *Light* to *Power* customers, you
> typically see (i) larger average basket sizes, (ii) higher reorder rates,
> and (iii) shorter intervals between orders. Together these paint a
> coherent picture of an increasingly habitual shopper as tenure grows.


### Average Reorder Rate by Customer Segment


Visualising just the reorder-rate column from the segment profile makes
the loyalty trend impossible to miss.


In [14]:
fig = px.bar(
    seg_profile,
    x="Segment",
    y="Avg Reorder Rate",
    title="Average Reorder Rate by Customer Segment",
    text_auto=".2f",
    color="Segment",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

> **Takeaway:** *Power* users have the highest reorder rate by a clear
> margin. They have learned what they like and stick to it — the perfect
> audience for a frictionless "buy your usuals" feature.


### Average Days Between Orders by Customer Segment


The flip side of reorder rate is **cadence**: how often does each segment
come back?


In [15]:
fig = px.bar(
    seg_profile,
    x="Segment",
    y="Avg Days Between Orders",
    title="Average Days Between Orders by Customer Segment",
    text_auto=".1f",
    color="Segment",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

> **Takeaway:** *Power* customers reorder roughly **every 1–2 weeks**;
> *Light* customers wait much longer between visits. Marketing should
> therefore use very different reactivation windows for the two groups —
> a 14-day silence means very different things for a Power vs a Light user.


### Customer Ordering Patterns: Day of Week × Hour of Day

A heatmap showing when customers place their orders.


We map the integer day-of-week codes to readable names and aggregate
order counts by `(day_of_week, hour_of_day)`.


In [ ]:
dow_map = {
    0: "Saturday",
    1: "Sunday",
    2: "Monday",
    3: "Tuesday",
    4: "Wednesday",
    5: "Thursday",
    6: "Friday",
}
orders_heatmap = orders.copy()
orders_heatmap["day_name"] = orders_heatmap["order_dow"].map(dow_map)

heatmap_data = (
    orders_heatmap.groupby(["day_name", "order_hour_of_day"])
    .size()
    .reset_index(name="count")
)
heatmap_data.head()

Pivot the long table into a 7×24 matrix (rows = days, columns = hours)
and plot it as a heatmap with `px.imshow`.


In [ ]:
pivot = heatmap_data.pivot(
    index="day_name", columns="order_hour_of_day", values="count"
)
pivot = pivot.reindex([dow_map[i] for i in range(7)])

fig = px.imshow(
    pivot,
    title="Order Volume by Day of Week and Hour of Day",
    labels=dict(x="Hour of Day", y="Day of Week", color="Orders"),
    aspect="auto",
    color_continuous_scale="Blues",
)
fig.show()

> **Takeaway:** The hottest cells are **Sunday and Saturday between
> 10 AM and 3 PM** — the prime weekend grocery-shopping window.
> Weekday peaks are flatter and shifted slightly later in the day.
> This is exactly the kind of view that should drive fulfilment-shift
> planning and notification scheduling.


### Basket Size vs Reorder Rate


Are customers who fill larger baskets also more loyal? We bin customers
by their average basket size and compute the mean reorder rate per bin.


In [17]:
# Bin basket sizes and compute average reorder rate per bin
customers_clean = customers.dropna(subset=["avg_basket_size", "reorder_rate"])
customers_clean = customers_clean.copy()
customers_clean["basket_bin"] = pd.cut(customers_clean["avg_basket_size"], bins=20)
basket_reorder = (
    customers_clean.groupby("basket_bin", observed=True)["reorder_rate"]
    .mean()
    .reset_index()
)
basket_reorder["basket_bin_mid"] = basket_reorder["basket_bin"].apply(lambda x: x.mid)

fig = px.line(
    basket_reorder,
    x="basket_bin_mid",
    y="reorder_rate",
    title="Average Reorder Rate by Basket Size",
    labels={
        "basket_bin_mid": "Average Basket Size",
        "reorder_rate": "Avg Reorder Rate",
    },
    markers=True,
)
fig.show()

> **Takeaway:** Reorder rate **rises with basket size** up to a point
> (roughly 15–20 items) and then plateaus. Customers doing real "weekly
> shop" baskets tend to be more loyal — large baskets are dominated by
> staples they always re-buy.


### Order Frequency vs Days Between Orders


A scatter plot of total orders against average days-between-orders, with
points coloured by segment and sized by basket size. We sample 5,000
customers so the scatter remains readable. Setting `random_state=42`
makes the sample reproducible.


In [18]:
# Sample 5,000 customers for readable scatter plot; random_state ensures reproducibility
cust_sample = customers.dropna(
    subset=["avg_days_between_orders", "avg_basket_size"]
).sample(n=min(5000, len(customers)), random_state=42)

fig = px.scatter(
    cust_sample,
    x="total_orders",
    y="avg_days_between_orders",
    color="frequency_segment",
    size="avg_basket_size",
    title="Order Frequency vs Average Days Between Orders (sample of 5,000 customers)",
    labels={
        "total_orders": "Total Orders",
        "avg_days_between_orders": "Avg Days Between Orders",
        "avg_basket_size": "Avg Basket Size",
        "frequency_segment": "Segment",
    },
    color_discrete_sequence=px.colors.qualitative.Set2,
    opacity=0.6,
    category_orders={"frequency_segment": seg_order},
)
fig.show()

> **Takeaway:** The expected hyperbolic shape is clear — more total orders
> ⇒ shorter gaps between orders. The colour overlay shows that the
> segments occupy distinct regions of the plot, validating the simple
> threshold-based segmentation.


### Department Preferences by Customer Segment


**Step 1 — Attach each user's segment to every item they bought.**


In [ ]:
user_seg = customers[["user_id", "frequency_segment"]].copy()
seg_dept = op_full.merge(user_seg, on="user_id")
seg_dept.head()

**Step 2 — Count items per (segment, department) and convert to within-segment proportions.**
We normalise *within* each segment so that, for example, the produce
share in the Light segment is comparable to the produce share in the
Power segment even though Power customers buy many more items overall.


In [ ]:
seg_dept_counts = (
    seg_dept.groupby(["frequency_segment", "department"])
    .size()
    .reset_index(name="item_count")
)

seg_totals = seg_dept_counts.groupby("frequency_segment")["item_count"].transform("sum")
seg_dept_counts["proportion"] = seg_dept_counts["item_count"] / seg_totals

# Keep only the top-10 departments overall so the chart stays readable
top10_depts = op["department"].value_counts().head(10).index.tolist()
seg_dept_top = seg_dept_counts[seg_dept_counts["department"].isin(top10_depts)]
seg_dept_top.head()

**Step 3 — Plot a stacked bar chart of department mix per segment.**


In [ ]:
fig = px.bar(
    seg_dept_top,
    x="frequency_segment",
    y="proportion",
    color="department",
    title="Department Proportion by Customer Segment (Top 10 Departments)",
    labels={
        "frequency_segment": "Customer Segment",
        "proportion": "Proportion of Items",
        "department": "Department",
    },
    barmode="stack",
    category_orders={"frequency_segment": seg_order},
)
fig.show()

> **Takeaway:** The department mix is remarkably **stable across
> segments** — produce, dairy/eggs, snacks and beverages dominate
> regardless of how often a customer orders. The shifts are subtle: as
> we move from *Light* to *Power*, the share of fresh and dairy categories
> creeps up, suggesting power-users lean even more heavily on essentials.


### How Reorder Rate Evolves Over Customer Lifetime


For every distinct `order_number` (1 = the customer's first order, n =
their nth order) we compute the average reorder rate across all customers
who reached that order number. This shows how *habitual* a customer's
purchases become over time.


In [20]:
# For orders in train set, link order_number
op_lifetime = order_products.merge(orders[["order_id", "order_number"]], on="order_id")

reorder_by_order_num = (
    op_lifetime.groupby("order_number")["reordered"].mean().reset_index()
)
reorder_by_order_num.columns = ["Order Number", "Reorder Rate"]
# Limit to first 50 orders for readability
reorder_by_order_num = reorder_by_order_num[reorder_by_order_num["Order Number"] <= 50]

fig = px.line(
    reorder_by_order_num,
    x="Order Number",
    y="Reorder Rate",
    title="Reorder Rate by Order Number in Customer Lifetime",
    markers=True,
)
fig.show()

> **Takeaway:** The first order has by definition a 0 % reorder rate
> (nothing has been bought before). The reorder rate then rises rapidly
> over the first ~5 orders before levelling off around **70–75 %**. In
> other words: once a customer has placed about five orders, their basket
> is dominated by familiar items — strong evidence that the early-tenure
> period is the critical window for shaping long-term habits.
